In [7]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'geopandas'

In [8]:
import pandas as pd

# Abrir y leer el archivo Excel
df_native = pd.read_excel('d_native_fix.xlsx')
print(df_native.columns)

Index(['Ciudad ', 'Provincia', 'ID', 'Latitud', 'Longitud', 'N Total',
       'Europa/MedioOriente', 'Africa', 'Nativos ', 'Originarios', 'A2', 'B2',
       'C1', 'D1'],
      dtype='object')


In [9]:
# Obtener información básica sobre el DataFrame
df_native.info()
# Mostrar las primeras filas del DataFrame para verificar que se ha leído correctamente
df_native.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Ciudad               20 non-null     object 
 1   Provincia            20 non-null     object 
 2   ID                   20 non-null     object 
 3   Latitud              20 non-null     float64
 4   Longitud             20 non-null     float64
 5   N Total              20 non-null     int64  
 6   Europa/MedioOriente  20 non-null     float64
 7   Africa               20 non-null     float64
 8   Nativos              20 non-null     float64
 9   Originarios          20 non-null     int64  
 10  A2                   20 non-null     float64
 11  B2                   20 non-null     float64
 12  C1                   20 non-null     float64
 13  D1                   20 non-null     float64
dtypes: float64(9), int64(2), object(3)
memory usage: 2.3+ KB


,Ciudad,Provincia,ID,Latitud,Longitud,N Total,Europa/MedioOriente,Africa,Nativos,Originarios,A2,B2,C1,D1
0,Sumampa,Santiago del Estero,SUM,-29.380936,-63.472556,29,3.4,0.0,96.6,28,0.214,0.071,0.321,0.393
1,Villa Atamisqui,Santiago del Estero,VA,-28.494608,-63.816292,56,3.6,3.6,96.6,52,0.173,0.096,0.519,0.212
2,Cordoba ciudad,Córdoba,cba,-31.420083,-64.188776,117,24.8,3.5,66.7,78,0.128,0.192,0.397,0.282


In [10]:
# Eliminar espacios en blanco alrededor de los nombres de las columnas
df_native.columns = df_native.columns.str.strip()
# Verificar nuevamente los nombres de las columnas
print(df_native.columns)

Index(['Ciudad', 'Provincia', 'ID', 'Latitud', 'Longitud', 'N Total',
       'Europa/MedioOriente', 'Africa', 'Nativos', 'Originarios', 'A2', 'B2',
       'C1', 'D1'],
      dtype='object')


In [13]:
# Transformacion tipos de datos: Ciudad, Provincia, ID a string.
df_native = df_native.astype({'Ciudad': 'string', 'Provincia': 'string', 'ID': 'string'})

### A CREAR MAPAS

In [12]:
import pandas as pd
import folium
import branca.colormap as cm

# Function to safely access dataframe columns
def safe_get(row, col, default="N/A"):
    return row[col] if col in row.index else default

# Calculate total genetic diversity
diversity_columns = ['Europa/MedioOriente', 'Africa', 'C1']
for col in diversity_columns:
    if col not in df_native.columns:
        print(f"Warning: Column '{col}' not found in dataframe. Skipping in diversity calculation.")
df_native['total_diversity'] = df_native[df_native.columns.intersection(diversity_columns)].sum(axis=1)

# Calculate the average coordinates
average_lat = df_native['Latitud'].mean()
average_lon = df_native['Longitud'].mean()

# Create a map centered on the average coordinates with a gray style
m = folium.Map(location=[average_lat, average_lon], zoom_start=6, tiles='CartoDB positron')

# Create a colormap for size based on 'N Total'
size_colormap = cm.LinearColormap(
    colors=['blue', 'green', 'yellow', 'red'],
    vmin=df_native['N Total'].min(),
    vmax=df_native['N Total'].max(),
    caption='Tamaño de la Muestra'
)

# Add the size colormap to the map
m.add_child(size_colormap)

# Add markers for each location
for idx, row in df_native.iterrows():
    try:
        folium.CircleMarker(
            location=[safe_get(row, 'Latitud'), safe_get(row, 'Longitud')],
            radius=safe_get(row, 'N Total', 1) / 10,  # Adjust size for visibility
            popup=f"""
            <strong style="font-size: 20px;">{safe_get(row, 'Ciudad')}, {safe_get(row, 'Provincia')}</strong><br>
            <strong>Información Completa:</strong><br>
            {row.to_frame().T.to_html(classes='table table-striped', header=True, index=False)}
            """,
            tooltip=safe_get(row, 'Ciudad'),  # Tooltip shows the name of the locality
            color=size_colormap(safe_get(row, 'N Total')),  # Color based on 'N Total'
            fill=True,
            fillColor=size_colormap(safe_get(row, 'N Total')),  # Fill color based on 'N Total'
            fillOpacity=0.7
        ).add_to(m)
    except Exception as e:
        print(f"Error adding marker for row {idx}: {e}")
        print(f"Row data: {row}")

# Add layer control
folium.LayerControl().add_to(m)

# Save the map
m.save("bauto_map.html")

print("El mapa ha sido guardado como 'bauto_map.html'")

# Display the map in the notebook
m


El mapa ha sido guardado como 'bauto_map.html'
